# Data Exploration and Preprocessing - Student Dataset Project
**Precious Oluwagbohunmi Faseyosan**

Steps Followed:
1. Load Data
2. Explore/Inspect Data
3. Handle Missing Values & Inconsistencies
4. Remove Duplicates
5. Detect and Treat Outliers
6. Scale and Transform Data
7. Encode Categorical Variables
8. Handle Noise
9. Reinspect Data

## Imports

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from scipy.stats import zscore

---
## Step 1: Load Data

In [ ]:
df = pd.read_excel('student_performance_raw.xlsx')
print('Dataset loaded successfully.')

---
## Step 2: Explore / Inspect Data

Before making any changes, we inspect the raw dataset to understand its structure,
data types, size, and any obvious quality problems.

In [ ]:
# Confirm the data loaded correctly and show the first 5 rows
df.head()

In [ ]:
# Display last 5 rows to check if data was cut off or if issues appear at the end
df.tail()

In [ ]:
# Show number of rows and columns
df.shape

In [ ]:
# Display column names
df.columns

In [ ]:
# Show data types, non-null counts, memory usage
df.info()

In [ ]:
# Show summary statistics for numerical columns
df.describe()

In [ ]:
# Count missing values per column
df.isnull().sum()

In [ ]:
# Unique values expose categorical inconsistencies invisible in describe().
# unique() catches variants like 'male', 'Male', 'M' treated as separate categories.

print('Gender unique values:     ', df['Gender'].unique())
print('Course unique values:     ', df['Course'].unique())
print('Final_Result unique values:', df['Final_Result'].unique())

---
## Step 3: Handle Missing Values & Inconsistencies

In [ ]:
# Two rows have text ages ('Twenty', 'Nineteen') - replace with integers.
# errors='coerce' converts remaining non-numeric values to NaN instead of crashing.

df['Age'] = df['Age'].replace({'Twenty': '20', 'Nineteen': '19'})
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

print('Non-numeric Age values remaining:', df['Age'].isna().sum())

In [ ]:
# Standardise Gender using str.title().str.strip() to fix casing and remove
# leading/trailing spaces. replace() replaces abbreviations with full labels.

df['Gender'] = df['Gender'].str.title().str.strip()
df['Gender'] = df['Gender'].replace({'M': 'Male', 'F': 'Female'})

print('Gender values after standardising:', df['Gender'].unique())

In [ ]:
# title().strip() collapses casing and spacing variants into 3 canonical categories.

df['Course'] = df['Course'].str.title().str.strip()

print('Course values after standardising:', df['Course'].unique())

In [ ]:
# Skewness analysis - determines whether mean or median is more appropriate for imputation.
numeric_cols = ['Age', 'Study_Hours', 'Attendance', 'Previous_GPA', 'Income']

# Method 1: skewness score
print('--- Skewness scores ---')
print(df[numeric_cols].skew())

# Method 2: compare mean vs median - close values indicate symmetry
print('\n--- Mean vs Median ---')
for col in numeric_cols:
    print(f"{col}: mean = {df[col].mean():.4f}, median = {df[col].median():.4f}")

In [ ]:
# Skewness between -0.5 and +0.5 across all columns - distributions are approximately symmetric.
# Mean imputation is valid here; median would produce the same result.

df['Study_Hours'] = df['Study_Hours'].fillna(df['Study_Hours'].mean())
df['Attendance']  = df['Attendance'].fillna(df['Attendance'].mean())

print('Missing values after imputation:')
print(df[['Study_Hours', 'Attendance']].isnull().sum())

In [ ]:
# Student S1454 has GPA = 0.0. Data shows a continuous low-GPA distribution (0.01, 0.02 ... 0.00).
# 0.0 is valid on a 0-4.0 scale - kept unchanged.

flagged_gpa = df[df['Previous_GPA'] == 0]
print('Flagged GPA: ')
print(flagged_gpa[['Student_ID', 'Previous_GPA']])
print('\nDecision: Kept as-is. Value is within the valid 0-4.0 range.')

In [ ]:
# Confirm all missing values have been handled
print('Missing values after Step 3:')
print(df.isnull().sum())

---
## Step 4: Remove Duplicates

In [ ]:
print('Total duplicate rows found:', df.duplicated().sum())

In [ ]:
# keep=False flags all copies including the first, so both versions appear side by side.

all_copies = df[df.duplicated(keep=False)].sort_values('Student_ID')
print(f'Showing all {len(all_copies)} rows involved in duplication:')
all_copies

In [ ]:
# Drop duplicates - keep the first occurrence of each duplicated record.

df = df.drop_duplicates()

print('Duplicates after removal:', df.duplicated().sum())
print('Rows remaining:', df.shape[0])

---
## Step 5: Detect and Treat Outliers (using Z-Score Method)

In [ ]:
# Z-score flags values >= 3 standard deviations from the mean as potential outliers.

print('--- Outlier detection (Z-score, threshold = 3) ---')
for col in numeric_cols:
    df['z_score'] = zscore(df[col])
    outliers = df[df['z_score'].abs() >= 3]
    print(f'{col}: {len(outliers)} outlier(s) detected')
    if len(outliers) > 0:
        print(outliers[['Student_ID', col]])

df.drop(columns=['z_score'], inplace=True)
print('\nNo outliers detected. No rows removed.')

---
## Export cleaned dataset (pre-transformation checkpoint)

In [ ]:
#Export cleaned dataset before transformation and making dataset ML ready

df.to_excel('student_performance_cleaned.xlsx', index=False)
print('Cleaned dataset (pre-transformation) saved to: student_performance_cleaned.xlsx')
print(f'Record count at this stage: {df.shape[0]} rows, {df.shape[1]} columns')

---
## Step 6: Scale and Transform Data (Min-Max Normalization)

In [ ]:
# Min-Max Scaling normalises numeric features to [0, 1].
# Chosen over Standard Scaling because Step 5 found zero outliers -
# no extreme values to handle, so the bounded [0, 1] range is appropriate.
# Scaled values stored in new columns (e.g., Age_Scaled) to preserve originals.

scaler = MinMaxScaler()

for col in numeric_cols:
    df[col + '_Scaled'] = scaler.fit_transform(df[[col]])

print('Scaled columns added.')

---
## Step 7: Encode Categorical Variables

In [ ]:
# Label Encoding for Gender and Final_Result - binary columns (2 categories only).
# Safe here because no false ordering is implied: Male/Female and Pass/Fail
# have no numerical relationship a model could misinterpret.
# Original text columns are dropped to avoid redundant columns in two formats.

le = LabelEncoder()

df['Gender_Encoded'] = le.fit_transform(df['Gender'])
print('Gender encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

df['Final_Result_Encoded'] = le.fit_transform(df['Final_Result'])
print('Final_Result encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

# Drop the original text columns now that encoded versions exist
df.drop(columns=['Gender', 'Final_Result'], inplace=True)

print('\nOriginal Gender and Final_Result columns dropped.')
print('Remaining columns:', df.columns.tolist())


In [ ]:
# One-Hot Encoding for Course - 3 categories with no natural order.
# Label Encoding would imply a false ranking (Statistics > Economics > Data Science).
# drop_first=True keeps 2 columns instead of 3 - the third is always implied,
# and dropping it avoids multicollinearity.

df = pd.get_dummies(df, columns=['Course'], drop_first=True)

# get_dummies creates boolean columns - convert to int (0/1) for clarity
course_cols = [col for col in df.columns if col.startswith('Course_')]
df[course_cols] = df[course_cols].astype(int)

print('One-Hot encoded Course columns:', course_cols)
print("Note: 'Data Science' is implicitly represented when both 'Course_Economics' and 'Course_Statistics' are 0.")
df[course_cols].head()

print('\nCurrent columns in DataFrame after encoding:', df.columns.tolist())


---
## Save Student_ID for linking

In [ ]:
# Save Student_ID before dropping so predictions can be traced back to individual students.
student_ids_for_linking = df[['Student_ID']].copy()
print('Student_ID column saved for linking:')
print(student_ids_for_linking.head())
print(f'Shape of saved student_ids_for_linking: {student_ids_for_linking.shape}')

---
## Drop Student_ID

Student_ID has no predictive value and can cause data leakage if treated as a feature.
Saved above for linking predictions back to students if needed.

In [ ]:
if 'Student_ID' in df.columns:
    df = df.drop(columns=['Student_ID'])
    print('Student_ID column dropped.')
else:
    print('Student_ID column not found or already dropped.')

print('Current columns in DataFrame:', df.columns.tolist())

---
## Step 8: Handle Noise

In [ ]:
# Domain-range checks flag logically impossible values, not statistical outliers.
# Attendance: 0-100%. Study_Hours: 0-10.

noisy_attendance = df[(df['Attendance'] > 100) | (df['Attendance'] < 0)]
print(f'Noisy Attendance values (outside 0-100): {len(noisy_attendance)} row(s)')

noisy_study = df[(df['Study_Hours'] > 10) | (df['Study_Hours'] < 0)]
print(f'Noisy Study_Hours values (outside 0-10): {len(noisy_study)} row(s)')

# Conditionally print the confirmation based on whether any noise was found
if len(noisy_attendance) == 0 and len(noisy_study) == 0:
    print('\nNo out-of-range values found. Dataset is clean on these fields.')
else:
    print('\nNoise was detected in the dataset on the above fields.')


---
## Step 9: Reinspect Data

In [ ]:
print('Final shape:', df.shape)

In [ ]:
print('Missing values in final dataset:')
print(df.isnull().sum())

In [ ]:
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
print('Gender_Encoded unique values:     ', df['Gender_Encoded'].unique())
print('Final_Result_Encoded unique values:', df['Final_Result_Encoded'].unique())

In [ ]:
print('Descriptive statistics of unscaled numeric columns:')
df[numeric_cols].describe().round(2)

In [ ]:
print('Sample of final cleaned dataset:')
df.head()

---
## Save Preprocessed Dataset

In [ ]:
df.to_excel('student_performance_ml_ready.xlsx', index=False)
print('ML-ready dataset saved to: student_performance_ml_ready.xlsx')
print(f'Final record count: {df.shape[0]} rows, {df.shape[1]} columns')